# Anomaly detection

Spot unusual events: failing machines, suspicious logins, odd transactions.

Two ways to start, from Hello Model: https://sayhellomodel.com/models/anomaly-detection/

**Your data:** the code uses example file names (such as `data.csv`). Upload your own file with the Files panel on the left, or change the name in the code. Run the cells from the top.

## Quick start

Statistical rules + Isolation Forest: Unsupervised, fast and easy to explain.

In [ ]:
import pandas as pd
df = pd.read_csv("sensor.csv", parse_dates=["ts"]).set_index("ts")
roll = df["temperature"].rolling("1h")
z = (df["temperature"] - roll.mean()) / roll.std()
alerts = df[z.abs() > 3]
print(alerts.head())

## Train your own model

Isolation Forest / ECOD ensembles via PyOD with rolling features: Strong unsupervised detectors with tunable alert volume.

In [ ]:
%pip install -q scikit-learn pandas

In [ ]:
# pip install scikit-learn pandas
import pandas as pd
from sklearn.ensemble import IsolationForest

df = pd.read_csv("sensor.csv", parse_dates=["ts"]).set_index("ts")
feats = pd.DataFrame({
    "mean_1h": df["temperature"].rolling("1h").mean(),
    "std_1h": df["temperature"].rolling("1h").std(),
    "vibration": df["vibration"],
}).dropna()

model = IsolationForest(n_estimators=300, contamination=0.005, random_state=42)
feats["anomaly"] = model.fit_predict(feats) == -1
feats["score"] = -model.score_samples(feats.drop(columns="anomaly"))
print(feats.sort_values("score", ascending=False).head(20))